# Seedance 2.0 Image-to-Video API 使用指南

本 Notebook 演示如何通过 **七牛 AIToken** 平台调用 **Seedance 2.0 (Image-to-Video)** 模型 API。

## 模型简介

Seedance 2.0 Image-to-Video 是字节跳动推出的图片转视频模型，支持以下能力：

- **起始帧动画**：将静态图片作为起始帧，根据提示词生成动态视频
- **首尾帧过渡**：支持同时指定起始帧和结束帧，生成平滑的过渡视频
- **同步音频生成**：可自动生成与视频同步的音频
- **多种宽高比**：支持 auto、21:9、16:9、4:3、1:1、3:4、9:16 等比例

**API 端点**：`bytedance/seedance-2.0/image-to-video`

## 1. 环境准备

首先安装 `fal-client` 依赖包，并设置 API Key 和七牛 AIToken 队列服务地址。

In [ ]:
# 安装 fal-client
%pip install fal-client -q

In [ ]:
import os

# 设置七牛 AIToken 队列服务地址
os.environ["FAL_QUEUE_RUN_HOST"] = "api.qnaigc.com/queue"

# 设置 FAL_KEY 环境变量
# 你也可以在终端中通过 export FAL_KEY="xxx" 来设置
os.environ["FAL_KEY"] = os.environ.get("QINIU_API_KEY") or (_ for _ in ()).throw(ValueError("环境变量 QINIU_API_KEY 未设置"))

In [ ]:
import fal_client
from IPython.display import Video, display

## 2. 基础用法 — 起始帧生成视频

七牛 AIToken 平台使用队列模式调用 API：先提交请求（`submit`），然后轮询状态（`status`），最后获取结果（`result`）。

下面先封装一个通用的调用辅助函数，后续示例都会复用。

In [ ]:
import time

def run_fal_task(endpoint, arguments, poll_interval=5):
    """提交任务并轮询等待结果返回"""
    # 步骤 1：提交请求
    handler = fal_client.submit(endpoint, arguments=arguments)
    request_id = handler.request_id
    print(f"请求已提交，request_id: {request_id}")

    # 步骤 2：轮询任务状态
    while True:
        status = fal_client.status(endpoint, request_id, with_logs=True)
        print(f"当前状态: {type(status).__name__}")

        if isinstance(status, fal_client.Completed):
            print("任务完成!")
            break

        if hasattr(status, "logs") and status.logs:
            for log in status.logs:
                print(log["message"])

        time.sleep(poll_interval)

    # 步骤 3：获取结果
    return fal_client.result(endpoint, request_id)


# 基础调用示例：起始帧 + 提示词
result = run_fal_task(
    "bytedance/seedance-2.0/image-to-video",
    arguments={
        "prompt": "An octopus finds a football in the ocean and excitedly calls its octopus friends to come and play. Cut scene to an octopus football game under the sea.",
        "image_url": "https://v3b.fal.media/files/b/0a8eba37/Cqg-4Uwzyz4DELfceT1CF_a17e588773ec45b1a9e6f100a787b80b.jpg",
    },
)

# 打印返回结果
print(result)

In [ ]:
# 展示生成的视频
video_url = result["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 3. 参数说明

### 可配置参数一览

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `prompt` | string | *必填* | 描述视频动作和运动的提示词 |
| `image_url` | string | *必填* | 起始帧图片 URL（支持 JPEG/PNG/WebP，≤30MB） |
| `end_image_url` | string | 可选 | 结束帧图片 URL，创建首尾帧之间的过渡效果 |
| `resolution` | enum | `720p` | 分辨率：`480p`、`720p` |
| `duration` | enum | `auto` | 视频时长：`auto` 或 `4`-`15` 秒 |
| `aspect_ratio` | enum | `auto` | 宽高比：`auto`、`21:9`、`16:9`、`4:3`、`1:1`、`3:4`、`9:16` |
| `generate_audio` | boolean | `true` | 是否生成与视频同步的音频 |
| `seed` | integer | 随机 | 随机种子，用于结果复现 |

## 4. 首尾帧过渡示例

通过同时指定 `image_url`（起始帧）和 `end_image_url`（结束帧），模型会生成两帧之间的平滑过渡视频。

In [ ]:
# 首尾帧过渡示例
result_transition = run_fal_task(
    "bytedance/seedance-2.0/image-to-video",
    arguments={
        "prompt": "第一人称视角果茶宣传广告，seedance牌「苹苹安安」苹果果茶限定款；你的手摘下一颗带晨露的阿克苏红苹果，轻脆的苹果碰撞声；第一人称手持举杯，你将果茶举到镜头前（模拟递到观众面前的视角），杯身标签清晰可见。",
        "image_url": "https://ark-project.tos-cn-beijing.volces.com/doc_image/r2v_tea_pic1.jpg",
        "end_image_url": "https://ark-project.tos-cn-beijing.volces.com/doc_image/r2v_tea_pic2.jpg",
        "duration": "8",
    },
)

# 展示结果
video_url = result_transition["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 5. 高级用法

使用更多可选参数来精细控制视频生成效果。

In [ ]:
# 高级用法：使用更多参数
result_advanced = run_fal_task(
    "bytedance/seedance-2.0/image-to-video",
    arguments={
        "prompt": "An octopus finds a football in the ocean and excitedly calls its octopus friends to come and play. Cut scene to an octopus football game under the sea.",
        "image_url": "https://v3b.fal.media/files/b/0a8eba37/Cqg-4Uwzyz4DELfceT1CF_a17e588773ec45b1a9e6f100a787b80b.jpg",
        "resolution": "720p",          # 720p 分辨率
        "duration": "10",              # 视频时长 10 秒
        "aspect_ratio": "16:9",        # 宽屏比例
        "generate_audio": True,         # 生成同步音频
        "seed": 42,                     # 固定随机种子，便于复现
    },
)

# 展示结果
video_url = result_advanced["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 6. 队列模式 — 手动分步调用

上面的 `run_fal_task` 封装了完整的队列流程。如果你需要更精细的控制（例如在提交后做其他事情，稍后再取结果），可以手动分步调用。

In [ ]:
# 步骤 1：提交请求
handler = fal_client.submit(
    "bytedance/seedance-2.0/image-to-video",
    arguments={
        "prompt": "An octopus finds a football in the ocean and excitedly calls its octopus friends to come and play. Cut scene to an octopus football game under the sea.",
        "image_url": "https://v3b.fal.media/files/b/0a8eba37/Cqg-4Uwzyz4DELfceT1CF_a17e588773ec45b1a9e6f100a787b80b.jpg",
        "aspect_ratio": "16:9",
    },
)

# 获取请求 ID，用于后续查询
request_id = handler.request_id
print(f"请求已提交，request_id: {request_id}")

In [ ]:
# 步骤 2：查询任务状态
import time

while True:
    status = fal_client.status(
        "bytedance/seedance-2.0/image-to-video",
        request_id,
        with_logs=True,
    )
    print(f"当前状态: {type(status).__name__}")

    if isinstance(status, fal_client.Completed):
        print("任务完成!")
        break

    # 每 5 秒查询一次
    time.sleep(5)

In [ ]:
# 步骤 3：获取结果
result_async = fal_client.result(
    "bytedance/seedance-2.0/image-to-video",
    request_id,
)

# 展示生成的视频
video_url = result_async["video"]["url"]
print(f"视频 URL: {video_url}")
display(Video(url=video_url, width=640))

## 7. Schema 参考

### Input Schema

```json
{
  "prompt": "string (必填)",
  "image_url": "string (必填, 起始帧图片 URL, 支持 JPEG/PNG/WebP, ≤30MB)",
  "end_image_url": "string (可选, 结束帧图片 URL)",
  "resolution": "480p | 720p (可选, 默认 720p)",
  "duration": "auto | 4-15 (可选, 默认 auto)",
  "aspect_ratio": "auto | 21:9 | 16:9 | 4:3 | 1:1 | 3:4 | 9:16 (可选, 默认 auto)",
  "generate_audio": "boolean (可选, 默认 true)",
  "seed": "integer (可选, 默认随机)"
}
```

### Output Schema

```json
{
  "video": {
    "url": "string (视频文件 URL)",
    "content_type": "string (MIME 类型)",
    "file_name": "string (文件名)",
    "file_size": "integer (文件大小)"
  },
  "seed": "integer (使用的随机种子)"
}
```

### 更多资源

- [七牛 AIToken 文档](https://developer.qiniu.com/aitokenapi)
- [fal-client Python 文档](https://docs.fal.ai/reference/client-libraries/python/fal_client)
- [Seedance 2.0 模型页面](https://fal.ai/models/bytedance/seedance-2.0)